In [99]:
%matplotlib auto
import cv2
import numpy as np
from IPython.display import display
from PIL import Image

Using matplotlib backend: inline


In [100]:
CHESS_ROW_SZ = 8
CHESS_COL_SZ = 8
AMT_CHESS_PIECE = 6

# piece index and maximum amount of the piece chess per side
PID_PAWN   = 0; MAX_AMT_PAWN   = 8
PID_ROOK   = 1; MAX_AMT_ROOK   = 2
PID_KNIGHT = 2; MAX_AMT_KNIGHT = 2
PID_BISHOP = 3; MAX_AMT_BISHOP = 2
PID_KING   = 4; MAX_AMT_KING   = 1
PID_QUEEN  = 5; MAX_AMT_QUEEN  = 1
PID_EMPTY  = 6; MAX_AMT_EMPTY  = 64

CID_WHITE = 0
CID_BLACK = 1

CAPTURE_POS = (CHESS_ROW_SZ, CHESS_COL_SZ)
MAXIMUM_WINDOW_SZ = 1024

M_TYPE_NO_MOVE = 0
M_TYPE_MOVE    = 1
M_TYPE_DIED    = 2

DUMMY_MOVE  = (M_TYPE_NO_MOVE, CHESS_ROW_SZ, CHESS_COL_SZ, CHESS_ROW_SZ, CHESS_COL_SZ, PID_EMPTY)

In [ ]:
class BoardData:
    # it contains current status of  the board
    # first two first dimension is board slot and the value is the piece chess type

    def __init__(self, param_confidentValue):
        self.confidentValue = param_confidentValue
        # current state
        self.curStates = np.zeros((2, CHESS_ROW_SZ, CHESS_COL_SZ), dtype=np.int8)
        ##### first for white and second for black
        self.nextFrames = 0
        self.curSize    = 0
        self.frames = np.zeros((2, MAXIMUM_WINDOW_SZ, CHESS_ROW_SZ, CHESS_COL_SZ, AMT_CHESS_PIECE + 1), dtype=np.int64)

    def addFrame(self, whiteFrame, blackFrame):
        
        self.frames[CID_WHITE][self.nextFrames] = whiteFrame
        self.frames[CID_BLACK][self.nextFrames] = blackFrame
        self.nextFrames = (self.nextFrames + 1) % MAXIMUM_WINDOW_SZ
        self.curSize    = min(self.curSize + 1, MAXIMUM_WINDOW_SZ)


    def vote(self, cid):

        frames_cid = self.frames[cid]

        voteBoard = np.zeros((CHESS_ROW_SZ, CHESS_COL_SZ, AMT_CHESS_PIECE + 1), dtype=np.int64)

        if (self.curSize > self.nextFrames):
            voteBoard = np.sum(frames_cid[:self.nextFrames], axis=0)
            remain    = self.curSize - self.nextFrames
            voteBoard = voteBoard + np.sum(frames_cid[-remain:], axis=0)
        else:
            voteBoard = np.sum(frames_cid[(self.nextFrames - self.curSize):self.nextFrames], axis=0)

        #print("vb", voteBoard)

        pieceResultIdx = np.argmax(voteBoard, axis=2)
        pieceResultVal = np.max   (voteBoard, axis=2)

        return pieceResultIdx, pieceResultVal
    
    def resetFrames(self):
        self.nextFrames = 0
        self.curSize    = 0


    def detectMove(self, cid, newState, newStateVoteVal): # return [missing from board] [appear]

        missingList = [] # (x,y, pid, voteValue)
        appearList  = [] # (x,y, pid, voteValue)

        oldState = self.curStates[cid]

        for r in range(CHESS_ROW_SZ):
            for c in range(CHESS_COL_SZ):
                #### detect missing
                if ((oldState[r][c] != PID_EMPTY) and (newState[r][c] == PID_EMPTY)):
                   missingList.append((r,c, oldState[r][c], newStateVoteVal[r][c]))
                #### detect appear
                if ((oldState[r][c] == PID_EMPTY) and (newState[r][c] != PID_EMPTY)):
                   appearList.append((r,c, newState[r][c], newStateVoteVal[r][c]))

        return missingList, appearList
    

    def updateState(self, cid, newState):
        self.curStates[cid] = newState


    #retur (moveType, src_x,src_y, des_x, des_y, pid)
    def analyzeMove(self, ml: list, al: list, cs):
        # sort with confident value
        sorted_ml = sorted(ml, key= lambda x : x[3], reverse=True)
        # sort with confident value
        sorted_al = sorted(al, key= lambda x : x[3], reverse=True)

        if (len(sorted_ml) == 0):
            return DUMMY_MOVE
        elif( (len(sorted_al) == 0) or 
             (sorted_al[0][2] != cs[sorted_ml[0][0]][sorted_ml[0][1]])):
            
            print("sorted_al[0][2] ", sorted_al)
            print("sorted_ml[0][2] ", cs)
            # it died
            return (M_TYPE_DIED, \
                    sorted_ml[0][0], sorted_ml[0][1], \
                    CHESS_ROW_SZ, CHESS_COL_SZ, sorted_al[0][2])
        else:
            print("moving")
            return (M_TYPE_MOVE, \
                    sorted_ml[0][0], sorted_ml[0][1], \
                    sorted_al[0][0], sorted_al[0][1], \
                    sorted_al[0][2]
                    )

        

    def screen(wm, bm):

        if (wm[0] == M_TYPE_MOVE) and (bm[0] == M_TYPE_DIED):
            return wm, bm
        
        if (wm[0] == M_TYPE_DIED) and (bm[0] == M_TYPE_MOVE):
            return wm, bm
        
        if (wm[0] == M_TYPE_MOVE) and (bm[0] == M_TYPE_NO_MOVE):
            return wm, bm
        
        if (wm[0] == M_TYPE_NO_MOVE) and (bm[0] == M_TYPE_MOVE):
            return wm, bm
        
        return DUMMY_MOVE, DUMMY_MOVE

In [102]:
def buildBoard(plist: list):

    board = np.zeros((CHESS_ROW_SZ, CHESS_ROW_SZ, AMT_CHESS_PIECE + 1), dtype=np.int8)
    board[:,:,PID_EMPTY] = 1
    for x,y,pid in plist:
        board[x][y][PID_EMPTY] = 0
        board[x][y][pid]     = 1
    return board

In [103]:
def testCase0():

    sys = BoardData(0.75)

    sys.curStates[CID_WHITE] = np.full((CHESS_ROW_SZ, CHESS_COL_SZ), PID_EMPTY, dtype=np.int8)
    sys.curStates[CID_WHITE][4][4] =PID_BISHOP
    sys.curStates[CID_BLACK] = np.full((CHESS_ROW_SZ, CHESS_COL_SZ), PID_EMPTY, dtype=np.int8)
    

    for _ in range(2000):
        startBoardWhite = buildBoard([(5,5,PID_BISHOP)])
        startBoardBlack = buildBoard([])
        sys.addFrame(startBoardWhite, startBoardBlack)
        #print(startBoardWhite)



    whitePieceIdxs, whitePieceVote = sys.vote(CID_WHITE)
    blackPieceIdxs, blackPieceVote = sys.vote(CID_BLACK)
    sys.resetFrames()
    print(whitePieceIdxs)
    wml, wal  = sys.detectMove(CID_WHITE, whitePieceIdxs, whitePieceVote)
    print(wml, wal)
    whiteMove = sys.analyzeMove(wml, wal, sys.curStates[CID_WHITE])
    bml, bal = sys.detectMove(CID_BLACK, blackPieceIdxs, blackPieceVote)
    blackMove = sys.analyzeMove(bml, bal, sys.curStates[CID_BLACK])

    print("whiteMove ", whiteMove)
    print("blackMove ", blackMove)

testCase0()

[[6 6 6 6 6 6 6 6]
 [6 6 6 6 6 6 6 6]
 [6 6 6 6 6 6 6 6]
 [6 6 6 6 6 6 6 6]
 [6 6 6 6 6 6 6 6]
 [6 6 6 6 6 3 6 6]
 [6 6 6 6 6 6 6 6]
 [6 6 6 6 6 6 6 6]]
[(4, 4, 3, 1024)] [(5, 5, 3, 1024)]
moving
whiteMove  (1, 4, 4, 5, 5, 3)
blackMove  (0, 8, 8, 8, 8, 6)
